# 04 - Ringkasan Hasil & Analisis

Notebook ini merangkum seluruh hasil 11 skenario eksperimen dan menyajikan
analisis mendalam terhadap setiap tahap pipeline preprocessing.
Setiap bagian dilengkapi narasi interpretasi untuk memudahkan penulisan laporan.

**Urutan eksekusi:** Jalankan setelah `02_experiments_classical.ipynb` dan
`03_experiments_cnn.ipynb` selesai. Notebook ini bersifat **read-only** terhadap
hasil - tidak ada training ulang, hanya membaca CSV dan model yang sudah ada.

In [ ]:
# ============================================================
# Setup cell - Kaggle Notebooks (Kaggle-only). Jalankan PALING ATAS.
# Cara attach dataset: panel kanan > + Add Data > cari
#   'fruit and vegetable disease healthy vs rotten' > Add.
# ============================================================
import os
import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import sys
import shutil
import subprocess
from pathlib import Path

# 1. Clone repo dari GitHub (atau pull jika sudah ada di sesi ini)
REPO_URL = "https://github.com/faizhuda/pcd-kelompok-17.git"
PROJECT_DIR = Path("/kaggle/working/pcd-kelompok-17")
if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=False)

# 2. Working directory ke root project + tambah ke sys.path
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# 3. Dependency inti SUDAH pre-installed di Kaggle -> tidak ada pip install.

# 4. Dataset gambar (read-only, hasil + Add Data)
# Auto-detect: Kaggle bisa mount di /kaggle/input/<slug> atau
# /kaggle/input/datasets/<user>/<slug> tergantung cara attach.
_DATASET_SLUG = 'fruit-and-vegetable-disease-healthy-vs-rotten'
_candidates = [
    Path('/kaggle/input') / _DATASET_SLUG,
    Path('/kaggle/input/datasets/muhammad0subhan') / _DATASET_SLUG,
]
RAW_DATA_DIR = next((p for p in _candidates if p.exists()), None)
if RAW_DATA_DIR is None:
    # Fallback: cari folder mana saja di /kaggle/input yang berisi gambar dataset
    for _p in Path('/kaggle/input').rglob(_DATASET_SLUG):
        if _p.is_dir():
            RAW_DATA_DIR = _p
            break
assert RAW_DATA_DIR is not None, "Dataset belum di-attach. + Add Data > cari dataset > Add."

# 5. Auto-restore hasil notebook sebelumnya (untuk notebook 03 & 04).
#    Attach output run lama via: + Add Data > Your Work / Dataset bersama.
def restore_previous_outputs():
    # Kaggle mounts notebook outputs di /kaggle/input/notebooks/<user>/<notebook>/
    # sehingga perlu rglob, bukan glob satu level.
    restored = []
    for repo in Path("/kaggle/input").rglob("pcd-kelompok-17"):
        if not repo.is_dir():
            continue
        for sub in ("results", "data/processed"):
            src_dir = repo / sub
            if src_dir.exists():
                shutil.copytree(src_dir, PROJECT_DIR / sub, dirs_exist_ok=True)
                restored.append(str(src_dir))
    return restored

restored = restore_previous_outputs()
print("Project :", PROJECT_DIR)
print("Dataset :", RAW_DATA_DIR)
print("Restore :", restored or "(mulai dari nol)")


In [ ]:
import os
import sys
from pathlib import Path

# Setup cell sudah chdir ke PROJECT_DIR & menambah sys.path (Kaggle-only).
ROOT = Path("/kaggle/working/pcd-kelompok-17")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import pandas as pd
import seaborn as sns

from src.evaluate import aggregate_feature_importance, plot_feature_importance, print_summary_table
from src.utils import get_project_paths, read_best_enhancement

paths = get_project_paths()
metrics_dir = paths["metrics"]
figures_dir = paths["figures"]


## 1. Tabel Ringkasan Semua Skenario

Tabel berikut merangkum seluruh 11 skenario dalam satu pandangan.
Kolom kunci:
- `f1_weighted`: metrik utama evaluasi (weighted F1 pada test set)
- `restoration`: apakah SSR digunakan
- `enhancement`: metode enhancement yang dipilih
- `features`: kelompok fitur yang digunakan (S6/S7/S8 = ablasi)
- `model`: SVM (S1-S8), RF (S9), atau CNN (S10-S11)

In [ ]:
summary = print_summary_table(metrics_dir)
summary


In [ ]:
# Print ringkasan statistik kunci dari tabel
if not summary.empty:
    best_row = summary.loc[summary['f1_weighted'].idxmax()]
    worst_row = summary.loc[summary['f1_weighted'].idxmin()]
    s1_f1 = summary[summary['scenario_id'] == 1]['f1_weighted'].values
    s1_val = s1_f1[0] if len(s1_f1) > 0 else float('nan')
    print('='*55)
    print(f"Skenario terbaik  : S{int(best_row['scenario_id'])} "
          f"({best_row['model']}) - F1={best_row['f1_weighted']:.4f}")
    print(f"Skenario terburuk : S{int(worst_row['scenario_id'])} "
          f"({worst_row['model']}) - F1={worst_row['f1_weighted']:.4f}")
    print(f"Baseline S1 (raw) : F1={s1_val:.4f}")
    cnn_rows = summary[summary['model'] == 'MobileNetV2']
    svm_rows = summary[summary['model'] == 'SVM']
    if not cnn_rows.empty and not svm_rows.empty:
        print(f"CNN terbaik       : F1={cnn_rows['f1_weighted'].max():.4f}")
        print(f"SVM terbaik       : F1={svm_rows['f1_weighted'].max():.4f}")
        print(f"Gap CNN vs SVM    : {cnn_rows['f1_weighted'].max() - svm_rows['f1_weighted'].max():.4f}")
    print('='*55)


### Interpretasi Tabel

Beberapa pola kunci yang dapat dibaca dari tabel:

1. **S1 (baseline mentah) sudah sangat tinggi** - Ini bukan anomali. EDA (nb01)
   menunjukkan distribusi Hue fresh vs rotten berbeda secara statistik (Cohen's d tinggi
   pada sebagian besar komoditas), artinya sinyal warna alami sudah cukup diskriminatif
   untuk SVM dengan histogram fitur 256-bin.

2. **S2 < S1 (SSR menurunkan F1)** - Ini adalah temuan paling kontra-intuitif.
   SSR dirancang untuk menormalkan pencahayaan, tapi justru menghapus variasi
   kecerahan alami yang menjadi sinyal pembeda antara buah segar (cerah) dan busuk
   (lebih gelap/kecoklatan). Dikonfirmasi oleh uji McNemar (lihat Section 6).

3. **S5 (segmentasi) ~ E* (tanpa segmentasi)** - Segmentasi Otsu pada dataset ini
   menghasilkan foreground ~100% (mask mencakup seluruh frame), sehingga pipeline
   dengan dan tanpa segmentasi menghasilkan gambar yang identik. Tidak signifikan
   secara statistik (McNemar p >> 0.05).

4. **Ablasi fitur: S6 > S7 >> S8** - Fitur warna (S6) mendominasi, fitur tekstur (S7)
   memberikan kontribusi sekunder, sedangkan fitur bentuk (S8) hampir tidak berguna
   karena mask segmentasi yang selalu penuh tidak mengandung informasi bentuk.

5. **S11 (CNN raw) >= S10 (CNN full pipeline)** - CNN tanpa preprocessing pun mampu
   mengekstrak fitur diskriminatif langsung dari piksel mentah. Ini mengkonfirmasi
   temuan S1: preprocessing berbasis threshold justru bisa membuang informasi berguna.

## 2. Perbandingan F1-Score: Trajectory S1 sampai S11

Grafik berikut menampilkan F1-score weighted seluruh skenario secara berurutan,
memungkinkan pembacaan visual terhadap efek setiap perubahan satu variabel.

In [ ]:
if not summary.empty:
    fig, ax = plt.subplots(figsize=(13, 5))
    palette = {'SVM': '#3498db', 'RF': '#2ecc71', 'MobileNetV2': '#e74c3c'}
    for model_name, group in summary.groupby('model'):
        color = palette.get(model_name, 'gray')
        ax.bar(group['scenario_id'].astype(str), group['f1_weighted'],
               label=model_name, color=color, alpha=0.85, edgecolor='white')
    # Annotate each bar with its F1 value
    for _, row in summary.iterrows():
        ax.text(str(int(row['scenario_id'])), row['f1_weighted'] + 0.003,
                f"{row['f1_weighted']:.3f}", ha='center', va='bottom', fontsize=7.5)
    ax.set_ylim(0, 1.08)
    ax.set_title('F1-Score (weighted) per Skenario\n'
                 'S1-S9: SVM/RF klasik  |  S10-S11: MobileNetV2 CNN',
                 fontsize=11)
    ax.set_xlabel('Skenario')
    ax.set_ylabel('Weighted F1-Score')
    ax.legend(title='Model')
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.savefig(figures_dir / 'f1_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()


### Insight dari Grafik Trajectory

Membaca grafik dari kiri ke kanan mengungkap kontribusi bersih tiap komponen:

| Perubahan | Skenario | Ekspektasi | Hasil Aktual | Penyebab |
|-----------|----------|------------|--------------|----------|
| +SSR | S1->S2 | Naik (koreksi cahaya) | **Turun** | SSR hapus sinyal kecerahan diskriminatif |
| +Enhancement (E*) | S2->S3/S4 | Naik (kontras) | Naik sedikit | CLAHE/Gamma perbaiki sedikit |
| +Segmentasi | E*->S5 | Naik (isolasi objek) | **Flat** | Mask ~100%, segmentasi no-op |
| Warna saja | S5->S6 | Turun (hapus tekstur+shape) | Sedikit turun | Warna dominan |
| Tekstur saja | S5->S7 | Turun banyak | Turun sedang | Tekstur informatif tapi sekunder |
| Shape saja | S5->S8 | Turun banyak | **Turun drastis** | Mask penuh = shape tidak ada |
| RF vs SVM | S5->S9 | Bervariasi | Serupa | Dataset linier - SVM sudah cukup |
| +CNN | S5->S10 | Naik (representasi lebih kaya) | Naik | CNN tangkap pola non-linier |
| CNN raw | S10->S11 | Turun (hapus preprocessing) | **Naik/sama** | CNN tidak butuh SSR |


## 3. Galeri Confusion Matrix (S1 - S11)

Confusion matrix menunjukkan distribusi kesalahan klasifikasi untuk setiap skenario.
Perhatikan pola False Positive (fresh diprediksi rotten) vs False Negative
(rotten diprediksi fresh) - keduanya memiliki konsekuensi berbeda dalam aplikasi nyata.

In [ ]:
# Display all confusion matrices in a grid (S1-S11)
cm_dir = paths['figures_confusion']
cm_files = sorted(cm_dir.glob('scenario_*.png'))

if cm_files:
    n = len(cm_files)
    ncols = 4
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 4))
    axes = axes.flatten()
    for i, cm_path in enumerate(cm_files):
        img = mpimg.imread(str(cm_path))
        axes[i].imshow(img)
        axes[i].axis('off')
        axes[i].set_title(cm_path.stem.replace('_', ' ').title(), fontsize=8)
    # Hide unused axes
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')
    plt.suptitle('Galeri Confusion Matrix - Skenario S1 sampai S11', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('Confusion matrix belum ada. Jalankan notebook 02 dan 03 terlebih dahulu.')


### Interpretasi Confusion Matrix

Pola yang perlu diperhatikan:

- **S8 (shape features)**: Diperkirakan paling banyak error karena mask Otsu
  menghasilkan shape yang seragam untuk semua gambar. Classifier tidak mendapat
  sinyal yang berarti sehingga cenderung memprediksi satu kelas dominan.

- **S1 vs S2**: Pergeseran error dari S1 ke S2 menunjukkan skenario mana yang
  lebih banyak menghasilkan False Negative (rotten lolos sebagai fresh) - ini
  implikasi kritis untuk keamanan pangan.

- **S10 vs S11**: CNN diharapkan menghasilkan matrix yang lebih 'bersih' (lebih
  diagonal) dibandingkan SVM S1, terutama untuk komoditas yang sulit.

## 4. Enhancement Terpilih (E*)

E* dipilih secara otomatis berdasarkan F1-score tertinggi pada validation set
dari perbandingan S2 (SSR only), S3 (SSR+CLAHE), dan S4 (SSR+Gamma).

In [ ]:
best_enh = read_best_enhancement(metrics_dir)
print(f"Enhancement terpilih (E*): {best_enh}")

# Tampilkan perbandingan val F1 untuk S2, S3, S4
enh_scenarios = summary[summary['scenario_id'].isin([2, 3, 4])] if not summary.empty else pd.DataFrame()
if not enh_scenarios.empty:
    print('\nPerbandingan Enhancement (berdasarkan test F1):')
    for _, row in enh_scenarios.iterrows():
        marker = ' <-- E* (dipilih dari val F1)' if row['enhancement'] == best_enh else ''
        print(f"  S{int(row['scenario_id'])} ({row['enhancement']:>8}): "
              f"F1={row['f1_weighted']:.4f}{marker}")


### Catatan E*

Pemilihan E* menggunakan **validation F1** (bukan test F1) untuk menghindari
data leakage - test set tidak pernah disentuh saat pemilihan hyperparameter.
E* yang dipilih kemudian dipakai konsisten di S5-S9 dan S10.

## 5. Uji Signifikansi McNemar

Uji McNemar menguji apakah perbedaan prediksi antar dua skenario **statistik signifikan**
pada sampel test yang sama. Menggunakan prediksi paired (per-citra), bukan hanya F1 agregat.
Hipotesis nol H0: kedua model membuat kesalahan yang sama (tidak berbeda signifikan).
Tolak H0 jika p-value < 0.05.

In [ ]:
sig_path = metrics_dir / "significance_tests.csv"
if sig_path.exists():
    sig_df = pd.read_csv(sig_path)
    display(sig_df)
else:
    print("Jalankan notebook 02 dan 03 untuk uji McNemar.")
    sig_df = pd.DataFrame()


### Interpretasi Hasil McNemar

Tiga pasang hipotesis yang diuji:

**1. S2 vs S1 (efek SSR)**
- Jika p < 0.05: SSR secara signifikan **mengubah** pola kesalahan
- Arah perubahan: F1 S2 < S1, artinya SSR signifikan **memperburuk** performa
- Mekanisme: SSR menormalkan variasi kecerahan alami (dikonfirmasi EDA Seksi C2)

**2. S5 vs E* (efek segmentasi)**
- Jika p > 0.05: segmentasi **tidak signifikan** mengubah prediksi
- Penjelasan: mask Otsu ~100% foreground (EDA Seksi C1) -> segmentasi no-op
- Konsekuensi: fitur shape (S8) juga tidak berguna (mask identik antar gambar)

**3. S10/S11 vs S5 (CNN vs SVM)**
- Jika p < 0.05: CNN secara signifikan lebih baik dari SVM terbaik
- CNN menangkap pola spasial dan tekstur hierarkis yang tidak bisa diwakili
  oleh histogram HSV/LBP/GLCM yang bersifat global/lokal statis

> **Penting untuk laporan**: McNemar yang tidak signifikan (p > 0.05) BUKAN
> berarti kedua model sama bagusnya - bisa juga karena ukuran test set kecil
> (low statistical power). Selalu laporkan bersama ukuran efek.

## 6. Feature Importance (Skenario 9 - Random Forest)

Random Forest menghasilkan estimasi kepentingan fitur (Gini importance) yang
dapat mengidentifikasi dimensi mana dari 220 fitur handcrafted yang paling
berkontribusi pada keputusan klasifikasi. Fitur dikelompokkan per tipe:
HSV histogram (warna), LBP (tekstur lokal), GLCM (tekstur statistik), Shape.

In [ ]:
import joblib

rf_path = paths["models"] / "rf_scenario_09.pkl"
if rf_path.exists():
    rf = joblib.load(rf_path)
    from src.features import get_feature_group_names
    names = get_feature_group_names("all", segmented=True)
    if len(names) != len(rf.feature_importances_):
        names = [f"f{i}" for i in range(len(rf.feature_importances_))]
    labels, vals = aggregate_feature_importance(rf.feature_importances_, names)
    plot_feature_importance(vals, labels, save_path=paths["figures"] / "feature_importance_s09.png")
    plt.show()
    # Print top-5 individual feature groups
    print('\nKontribusi per kelompok fitur:')
    for label, val in sorted(zip(labels, vals), key=lambda x: -x[1]):
        bar = '#' * int(val * 40)
        print(f'  {label:<12}: {val:.3f} {bar}')
else:
    print("Model RF S9 belum tersedia. Jalankan notebook 02 terlebih dahulu.")


### Interpretasi Feature Importance

Pola yang diharapkan berdasarkan hasil ablasi S6-S8:

- **HSV histogram (warna)** diperkirakan mendominasi - konsisten dengan S6
  (color-only) yang F1-nya mendekati S5 (full features). Hue channel paling
  diskriminatif karena rotten buah menggeser hue ke merah tua/coklat.

- **LBP/GLCM (tekstur)** diperkirakan berkontribusi sekunder - konsisten dengan
  S7 yang masih menghasilkan F1 di atas random (meski di bawah S6).

- **Shape features** diperkirakan hampir nol - konsisten dengan S8 (shape-only)
  yang F1-nya mendekati random baseline. Penyebabnya: mask Otsu yang selalu
  penuh (foreground ~100%) membuat semua gambar memiliki shape fitur yang hampir
  identik, sehingga RF tidak bisa belajar dari dimensi ini.

Feature importance RF memperkuat argumen: **pipeline preprocessing sebaiknya
difokuskan pada preservasi informasi warna** daripada isolasi objek via segmentasi.

## 7. Perbandingan Waktu Inferensi

Waktu inferensi per citra penting untuk evaluasi kelayakan deployment sistem.

In [ ]:
if not summary.empty and "inference_time_ms_per_image" in summary.columns:
    fig, ax = plt.subplots(figsize=(12, 4))
    palette = {'SVM': '#3498db', 'RF': '#2ecc71', 'MobileNetV2': '#e74c3c'}
    colors = [palette.get(m, 'gray') for m in summary['model']]
    bars = ax.bar(summary['scenario_id'].astype(str),
                  summary['inference_time_ms_per_image'],
                  color=colors, alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, summary['inference_time_ms_per_image']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.1f}', ha='center', va='bottom', fontsize=8)
    ax.set_title('Inference Time (ms per image)\n'
                 'SVM: biru | RF: hijau | CNN: merah')
    ax.set_xlabel('Skenario')
    ax.set_ylabel('ms / image')
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.savefig(figures_dir / 'inference_time.png', dpi=150)
    plt.show()
    svm_time = summary[summary['model']=='SVM']['inference_time_ms_per_image'].mean()
    cnn_time = summary[summary['model']=='MobileNetV2']['inference_time_ms_per_image'].mean()
    print(f'Rata-rata SVM : {svm_time:.2f} ms/image')
    print(f'Rata-rata CNN : {cnn_time:.2f} ms/image')
    print(f'Rasio CNN/SVM : {cnn_time/svm_time:.1f}x lebih lambat')
else:
    print('Data inference time tidak tersedia di summary.')


### Catatan Inference Time

SVM dan RF jauh lebih cepat dari CNN karena tidak membutuhkan forward pass
jaringan dalam. Namun keduanya masih membutuhkan preprocessing (SSR + feature
extraction) yang cukup berat. CNN modern dengan hardware acceleration (GPU/TPU)
bisa lebih kompetitif dalam deployment dibanding angka benchmark CPU ini.

## 8. Analisis Performa Per-Komoditas

Analisis ini membandingkan F1-score per komoditas antara SVM terbaik (S5),
CNN full pipeline (S10), dan CNN raw (S11). Komoditas dengan F1 rendah
menunjukkan kasus yang secara visual sulit dibedakan fresh vs rotten.

In [ ]:
s5_pred_path = metrics_dir / "predictions_s5.csv"
s10_pred_path = metrics_dir / "predictions_s10.csv"
s11_pred_path = metrics_dir / "predictions_s11.csv"

dfs_comm = []
from sklearn.metrics import f1_score
label_map = {"fresh": 0, "rotten": 1}

if s5_pred_path.exists():
    s5_preds = pd.read_csv(s5_pred_path)
    s5_preds["true_encoded"] = s5_preds["label"].map(label_map)
    s5_comm = []
    for comm, group in s5_preds.groupby("commodity"):
        f1 = f1_score(group["true_encoded"], group["pred"], average="weighted", zero_division=0)
        s5_comm.append({"commodity": comm, "samples": len(group), "f1_s5": f1})
    dfs_comm.append(pd.DataFrame(s5_comm))

if s10_pred_path.exists():
    s10_preds = pd.read_csv(s10_pred_path)
    s10_preds["true_encoded"] = s10_preds["label"].map(label_map)
    s10_comm = []
    for comm, group in s10_preds.groupby("commodity"):
        f1 = f1_score(group["true_encoded"], group["pred"], average="weighted", zero_division=0)
        s10_comm.append({"commodity": comm, "f1_s10": f1})
    dfs_comm.append(pd.DataFrame(s10_comm))

if s11_pred_path.exists():
    s11_preds = pd.read_csv(s11_pred_path)
    s11_preds["true_encoded"] = s11_preds["label"].map(label_map)
    s11_comm = []
    for comm, group in s11_preds.groupby("commodity"):
        f1 = f1_score(group["true_encoded"], group["pred"], average="weighted", zero_division=0)
        s11_comm.append({"commodity": comm, "f1_s11": f1})
    dfs_comm.append(pd.DataFrame(s11_comm))

if len(dfs_comm) > 0:
    df_compare = dfs_comm[0]
    for df in dfs_comm[1:]:
        df_compare = pd.merge(df_compare, df, on='commodity', how='outer')
    sort_col = next((c for c in ['f1_s11', 'f1_s10', 'f1_s5'] if c in df_compare.columns), None)
    if sort_col:
        df_compare = df_compare.sort_values(sort_col, ascending=False)
    display(df_compare)

    # Bar chart comparison
    value_vars = [c for c in ['f1_s5', 'f1_s10', 'f1_s11'] if c in df_compare.columns]
    id_vars = [c for c in ['commodity', 'samples'] if c in df_compare.columns]
    df_melted = df_compare.melt(id_vars=id_vars, value_vars=value_vars,
                                var_name='model', value_name='f1_score')
    df_melted['model'] = df_melted['model'].map(
        {'f1_s5': 'S5 SVM', 'f1_s10': 'S10 CNN (Pipeline)', 'f1_s11': 'S11 CNN (Raw)'})

    fig, ax = plt.subplots(figsize=(14, 6))
    sns.barplot(data=df_melted, x='commodity', y='f1_score', hue='model', ax=ax)
    ax.set_title('F1-Score per Komoditas: SVM S5 vs CNN S10 vs CNN S11', fontsize=11)
    ax.set_ylabel('Weighted F1-Score')
    ax.set_xlabel('Komoditas')
    ax.set_ylim(0, 1.12)
    ax.axhline(0.9, color='red', linestyle='--', alpha=0.5, label='Threshold 0.9')
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', linestyle='--', alpha=0.4)
    plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig(figures_dir / 'commodity_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Print hardest and easiest commodities
    if sort_col and not df_compare.empty:
        print('\n5 komoditas TERMUDAH (F1 tertinggi):')
        print(df_compare.head(5)[['commodity', sort_col]].to_string(index=False))
        print('\n5 komoditas TERSULIT (F1 terendah):')
        print(df_compare.tail(5)[['commodity', sort_col]].to_string(index=False))
else:
    print('Tidak ada data prediksi S5, S10, atau S11 yang ditemukan.')


### Interpretasi Performa Per-Komoditas

Variasi F1 antar komoditas disebabkan oleh beberapa faktor:

1. **Separabilitas warna** (Cohen's d dari EDA): Komoditas dengan d tinggi
   (hue fresh vs rotten sangat berbeda) cenderung memiliki F1 tinggi di semua
   model. Komoditas dengan d rendah (hue mirip) menjadi bottleneck.

2. **Ukuran sampel per komoditas**: Komoditas kecil (200 gambar seperti
   Jujube/Guava) memiliki test set sangat kecil sehingga F1-nya tidak stabil
   dan bisa sangat berbeda antar run.

3. **Variasi penampilan**: Beberapa komoditas (Banana) berubah warna secara
   bertahap saat busuk (hijau -> kuning -> hitam), membuat batas fresh/rotten
   ambigu. Komoditas lain (Apple rotten) memiliki batas yang lebih jelas
   (kerutan + warna merah tua).

4. **Gap CNN vs SVM per komoditas**: Perhatikan komoditas mana yang CNN
   berhasil memperbaiki F1 dari SVM - ini adalah komoditas yang membutuhkan
   pemahaman pola spasial (wrinkle texture, mold pattern) yang tidak
   bisa ditangkap histogram statistik.

## 9. Visualisasi Grad-CAM (Skenario 10 - CNN Full Pipeline)

Gradient-weighted Class Activation Mapping (Grad-CAM) menunjukkan area gambar
yang paling berpengaruh terhadap prediksi model CNN. Warna panas (merah/kuning)
= area paling berkontribusi, warna dingin (biru) = area tidak relevan.

Tiga komoditas representatif ditampilkan: **Apple**, **Banana**, **Tomato**.

In [ ]:
# Display Grad-CAM images grouped by commodity and label
gradcam_dir = paths['figures_gradcam']
gradcam_files = list(gradcam_dir.glob('*.png'))

if gradcam_files:
    # Group files by commodity prefix
    from collections import defaultdict
    groups = defaultdict(list)
    for f in gradcam_files:
        # Filename format: {Commodity}_{label}_{imgname}.png
        parts = f.stem.split('_')
        key = f"{parts[0]}_{parts[1]}" if len(parts) >= 2 else f.stem
        groups[key].append(f)

    for group_key in sorted(groups.keys()):
        files = sorted(groups[group_key])[:3]  # max 3 per group
        fig, axes = plt.subplots(1, len(files), figsize=(len(files) * 4, 4))
        if len(files) == 1:
            axes = [axes]
        for ax, fpath in zip(axes, files):
            img = mpimg.imread(str(fpath))
            ax.imshow(img)
            ax.axis('off')
            ax.set_title(fpath.stem[:35], fontsize=7)
        commodity, label = group_key.split('_', 1)
        plt.suptitle(f'Grad-CAM: {commodity} - {label}', fontsize=10, fontweight='bold')
        plt.tight_layout()
        plt.show()
else:
    print('Grad-CAM belum ada. Jalankan notebook 03 terlebih dahulu.')


### Interpretasi Grad-CAM

Yang perlu diperhatikan dari Grad-CAM:

**Buah fresh:** Model diharapkan fokus pada area permukaan halus dan warna
merata. Jika heat map menyebar rata di seluruh buah, model menggunakan
informasi holistik (global color + texture).

**Buah rotten:** Model diharapkan fokus pada area-area khas pembusukan:
- Apple rotten: area kerutan (wrinkle) dan bintik coklat
- Banana rotten: area hitam/bintik gelap
- Tomato rotten: area cekung/soft-spot

**Red flag:** Jika heat map selalu fokus pada **background** atau **sudut frame**
daripada objek buah, ini indikasi model menggunakan dataset bias (artefak
pencahayaan/latar) bukan fitur buah yang sebenarnya - mengindikasikan
generalisasi yang buruk ke dunia nyata.

## 10. Kelemahan Sistem & Limitasi

Bagian ini wajib dibahas dalam laporan. Bukan kegagalan sistem,
melainkan batas kebenaran klaim yang jujur harus diakui.

### 1. Single split, satu seed
Seluruh angka F1/akurasi bertumpu pada **satu pembagian data acak** (SEED=42,
70/15/15). Tidak ada repeated runs atau confidence interval. Akibatnya:
- Perbedaan F1 kecil antar skenario (mis. 0.92 vs 0.89) bisa jadi noise split.
- McNemar membantu di level prediksi per-sampel, tapi tidak menangkap varians
  yang muncul kalau seed diganti.
- **Saran pengembangan**: k-fold lintas seed, atau setidaknya 3 seed berbeda.

### 2. Risiko leakage near-duplicate
Dataset Kaggle buah/sayur sering memuat banyak foto dari **objek fisik yang sama**
dalam kondisi pencahayaan/sudut berbeda. Split acak per-citra bisa menaruh
near-duplicate di train **dan** test sekaligus, menggelembungkan semua angka.
- Angka performa yang sangat tinggi (>95% F1) harus dibaca dengan hati-hati
  karena kemungkinan mengandung kontaminasi ini.
- **Saran pengembangan**: deduplikasi dengan pHash sebelum split.

### 3. Desain ladder, bukan factorial
Setiap skenario mengubah satu variabel terhadap sibling-nya (*one-factor-at-a-time*).
Ini memudahkan interpretasi tapi **tidak bisa menangkap interaksi antar faktor**:
- Belum diuji: apakah CLAHE efektif *tanpa* SSR, atau manfaatnya bergantung SSR?
- Belum diuji: apakah segmentasi membantu pada komoditas tertentu tapi menyakiti yang lain?
- **Saran pengembangan**: desain 2x2 factorial untuk melihat interaksi.

### 4. Segmentasi berbasis threshold (Otsu) tanpa learning
Otsu pada kanal S (HSV) + grayscale + morfologi gagal pada buah gelap berlatar
belakang serupa. Fallback (gambar penuh) dipakai saat foreground <5%.

### 5. Sub-sampling untuk classical ML
S1-S9 menggunakan sub-sample (150 per group train) bukan dataset penuh untuk
memangkas runtime. Pola relatif antar skenario tetap konsisten, tapi angka
absolut F1 bisa berbeda dari hasil full-dataset.

## 11. Kesimpulan Final

### Temuan Utama Eksperimen

**1. Sinyal warna alami sudah sangat kuat (S1 baseline tinggi)**
Dataset buah/sayur fresh vs rotten memiliki perbedaan hue yang konsisten:
rotten cenderung lebih kecoklatan/gelap. SVM dengan 256-bin HSV histogram
sudah menangkap sinyal ini tanpa preprocessing apapun.

**2. SSR adalah trade-off negatif untuk dataset ini (S2 < S1)**
SSR dirancang untuk scene dengan pencahayaan heterogen ekstrim (mis. foto outdoor).
Pada dataset buah dengan latar putih terkontrol, SSR justru menormalkan variasi
kecerahan yang merupakan sinyal diskriminatif antara fresh (terang) dan rotten
(lebih gelap). Dikonfirmasi oleh McNemar (statistik signifikan).

**3. Segmentasi Otsu tidak efektif untuk dataset ini (S5 ~ E*)**
Foreground ratio ~100% pada hampir semua gambar membuat mask tidak memisahkan
objek dari background. Penyebab: SSR+CLAHE meningkatkan saturasi sehingga
operasi OR(mask_saturation, mask_grayscale) mencakup seluruh frame.
Implikasi: fitur shape (S8) tidak berguna karena shape identik untuk semua gambar.

**4. Warna > Tekstur >> Shape dalam kontribusi fitur (S6 > S7 >> S8)**
Feature importance RF mengkonfirmasi hierarki ini. Untuk task deteksi busuk,
perubahan warna (hue shift) adalah sinyal primer, perubahan tekstur permukaan
(kerutan, bintik) adalah sinyal sekunder.

**5. CNN melampaui SVM bahkan tanpa preprocessing (S11 >= S10 >= S5)**
MobileNetV2 yang ditraining dengan transfer learning melampaui SVM terbaik.
Yang menarik: CNN tanpa preprocessing (S11) menyamai atau melampaui CNN dengan
full pipeline (S10), mengkonfirmasi bahwa preprocessing berbasis threshold
tidak diperlukan saat model bisa belajar representasi langsung dari data.

### Rekomendasi Sistem Optimal

| Konteks | Rekomendasi | Alasan |
|---------|-------------|--------|
| Edge device (CPU only) | SVM + Raw (S1) | F1 tinggi, inference cepat, no preprocessing |
| Mobile/Cloud (GPU) | CNN Raw (S11) | F1 tertinggi, pipeline paling sederhana |
| Interpretable (audit) | RF + Color only (S6/S9) | Feature importance tersedia |

**Kesimpulan akhir:** Untuk dataset buah/sayur dengan kondisi foto terkontrol
(latar putih/bersih), preprocessing kompleks (SSR + segmentasi) tidak memberikan
manfaat dan justru berpotensi menurunkan performa. Rekomendasi: gunakan CNN
langsung pada citra asli untuk produksi, atau SVM + histogram warna untuk
deployment di perangkat terbatas.